In [43]:
import re
import numpy as np
import pandas as pd
import requests, os, json
from dotenv import load_dotenv
from collections import defaultdict
from datetime import datetime

In [4]:
load_dotenv()

True

In [5]:
TOKEN = os.getenv("TOKEN")
OWNER_TYPE = "organization"
OWNER_NAME = os.getenv("OWNER_NAME")
PROJECT_NUMBER = int(os.getenv("PROJECT_NUMBER"))

In [7]:
query = """
query($owner: String!, $number: Int!) {
  viewer {
    id
  }
  %s(login: $owner) {
    projectV2(number: $number) {
      items(first: 100) {
        nodes {
          content {
            ... on Issue {
              id
              title
              url
              author {
                login
              }
              createdAt
              updatedAt
              state
              assignees(first: 10) {
                nodes {
                  login
                }
              }
              labels(first: 10) {
                nodes {
                  name
                }
              }
            }
          }
          fieldValues(first: 10) {
            nodes {
              ... on ProjectV2ItemFieldSingleSelectValue {
                name
                field {
                  ... on ProjectV2FieldCommon {
                    name
                  }
                }
              }
            }
          }
        }
      } 
    }
  }
}
""" % OWNER_TYPE

In [8]:
headers = {"Authorization": f"Bearer {TOKEN}"}
variables = {"owner": OWNER_NAME, "number": PROJECT_NUMBER}

# Execute the request
response = requests.post(
    "https://api.github.com/graphql",
    headers=headers,
    json={"query": query, "variables": variables}
)


KeyboardInterrupt: 

## Parsers

### Bronze Parser

In [22]:
def _get_df(data, status):
  df = pd.DataFrame(data.get(status))
  df = _add_status_to_df(status, df)
  return df

def _add_status_to_df(status: str, df: pd.DataFrame):
  df['status'] = status.lower()
  return df

def transform_raw_to_bronze(data: dict, extracted_at: datetime = None):

    if extracted_at is None:
        extracted_at = datetime.utcnow()

    columns = defaultdict(list)

    items = data.get("data", {}).get(OWNER_TYPE, {}).get("projectV2", {}).get("items", {}).get("nodes", [])

    for item in items:

        if not item.get("content") or "url" not in item["content"]:
            continue

        issue_url = item["content"]["url"]
        issue_id = item["content"]["id"] 
        issue_title = item["content"].get("title", "Untitled")

        author_data = item["content"].get("author")
        username = author_data.get("login") if author_data else "Unknown User"

        created_at = item["content"].get("createdAt", "Unknown")
        updated_at = item["content"].get("updatedAt", "Unknown")
        state = item["content"].get("state", "Unknown")

        assignees_data = item["content"].get("assignees", {}).get("nodes", [])
        assignees = [user.get("login") for user in assignees_data if user]

        labels_data = item["content"].get("labels", {}).get("nodes", [])
        labels_data = [label.get("name") for label in labels_data if label]

        labels = item["content"].get("labels", {}).get("nodes", [])
        
        milestone = None
        if labels:
            for label in labels:
                label = label.get("name")
                if label.startswith("M") and label[1:].isdigit():
                    milestone = label
    
        if milestone == None:
            re.search(r"\[(M\d+)\]", issue_title)        
    
        status = "No Status"
    
        for field in item.get("fieldValues", {}).get("nodes", []):
            if not field:
                continue
            if field.get("field", {}).get("name") == "Status":
                status = field["name"]
                break
    
        columns[status].append({
            "id": issue_id,
            "title": issue_title,
            "url": issue_url,
            "author": username,
            "created_at": created_at,
            "updated_at": updated_at,
            "state": state,
            "assignees": assignees,
            "labels": labels_data,
            "extracted_at": extracted_at,
            "milestone": milestone
        })
    
    ni_df = _get_df(columns, 'Needs Improvement')
    p_df = _get_df(columns, 'Passed')
    ir_df = _get_df(columns, 'In review')
    u_df = _get_df(columns, 'Unchecked/Unsigned')
    
    bronze_table = pd.concat([ni_df, p_df, ir_df, u_df], ignore_index=True)
    bronze_table.rename(
        columns={
            "id": "issue_id"
        }, inplace=True
    )
    
    return bronze_table[bronze_table["title"].str.match(r"\[M\d+\]")]

In [18]:
with open("../data/v2-response.json", 'r') as fp:
    data = json.load(fp)
    print(type(data))

<class 'dict'>


In [23]:
bronze_table = transform_raw_to_bronze(data)
bronze_table.head()

/tmp/ipykernel_22640/1760413641.py:13: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  extracted_at = datetime.utcnow()


,issue_id,title,url,author,created_at,updated_at,state,assignees,labels,extracted_at,milestone,status
0,I_kwDOShte388AAAABIWAL1g,[M1] Mark Ivan Contemprato — Weekly Fuel Price...,https://github.com/dataengineeringpilipinas/de...,navikram03,2026-07-10T12:18:57Z,2026-07-30T12:26:36Z,OPEN,[nicholailim],"[milestone-submission, needs-improvement, M1]",2026-07-31 16:09:44.146876,M1,needs improvement
1,I_kwDOShte388AAAABIQvtWQ,[M1] Jerome Fabrero — Philippine Wholesale Ele...,https://github.com/dataengineeringpilipinas/de...,jffabrero,2026-07-09T19:46:39Z,2026-07-29T07:54:07Z,OPEN,[kimodri],"[milestone-submission, M1, ready-for-review]",2026-07-31 16:09:44.146876,M1,needs improvement
2,I_kwDOShte388AAAABH9Hx5w,[M1] supjkay — Beating the Rush: The Metro Man...,https://github.com/dataengineeringpilipinas/de...,supjkay,2026-07-07T13:09:48Z,2026-07-24T14:38:25Z,OPEN,[webzero13],"[milestone-submission, needs-improvement, M1]",2026-07-31 16:09:44.146876,M1,needs improvement
3,I_kwDOShte388AAAABG_LhBA,[M0] Ken Shamrock Dizon — How has bird species...,https://github.com/dataengineeringpilipinas/de...,ksdizon,2026-06-28T22:49:06Z,2026-07-08T17:28:10Z,OPEN,[kimodri],"[milestone-submission, M0]",2026-07-31 16:09:44.146876,M0,needs improvement
4,I_kwDOShte388AAAABGT6ozQ,[M0] Cyan — The AI Divide in Computer Studies ...,https://github.com/dataengineeringpilipinas/de...,penOnFire,2026-06-22T16:17:50Z,2026-07-21T16:39:13Z,OPEN,[webzero13],"[milestone-submission, needs-improvement, M0]",2026-07-31 16:09:44.146876,M0,needs improvement


In [47]:
# specify the type of date
# handle multi-reviewers // explode
# drop labels

silver_table = bronze_table

date_cols = ["created_at", "updated_at", "extracted_at"]

for date_col in date_cols:
    silver_table[date_col] = (
        pd.to_datetime(silver_table[date_col], utc=True)
        .dt.tz_convert("Asia/Manila")
        .dt.floor("s")
    )

silver_table_exploded = silver_table.explode("assignees")\
    .drop(columns=["labels"])



In [48]:
silver_table_exploded["is_assigned"] = 1 if (silver_table_exploded["assignees"] is not None) else 0
silver_table_exploded["days_since_update"] = (
    silver_table_exploded["updated_at"] - silver_table_exploded["created_at"]
).dt.days

silver_table_exploded["submission_age_days"] = (
    silver_table_exploded["extracted_at"] - silver_table_exploded["created_at"]
).dt.days


silver_table_exploded.head()

,issue_id,title,url,author,created_at,updated_at,state,assignees,extracted_at,milestone,status,is_assigned,days_since_update,submission_age_days
0,I_kwDOShte388AAAABIWAL1g,[M1] Mark Ivan Contemprato — Weekly Fuel Price...,https://github.com/dataengineeringpilipinas/de...,navikram03,2026-07-10 20:18:57+08:00,2026-07-30 20:26:36+08:00,OPEN,nicholailim,2026-08-01 00:09:44+08:00,M1,needs improvement,1,20,21
1,I_kwDOShte388AAAABIQvtWQ,[M1] Jerome Fabrero — Philippine Wholesale Ele...,https://github.com/dataengineeringpilipinas/de...,jffabrero,2026-07-10 03:46:39+08:00,2026-07-29 15:54:07+08:00,OPEN,kimodri,2026-08-01 00:09:44+08:00,M1,needs improvement,1,19,21
2,I_kwDOShte388AAAABH9Hx5w,[M1] supjkay — Beating the Rush: The Metro Man...,https://github.com/dataengineeringpilipinas/de...,supjkay,2026-07-07 21:09:48+08:00,2026-07-24 22:38:25+08:00,OPEN,webzero13,2026-08-01 00:09:44+08:00,M1,needs improvement,1,17,24
3,I_kwDOShte388AAAABG_LhBA,[M0] Ken Shamrock Dizon — How has bird species...,https://github.com/dataengineeringpilipinas/de...,ksdizon,2026-06-29 06:49:06+08:00,2026-07-09 01:28:10+08:00,OPEN,kimodri,2026-08-01 00:09:44+08:00,M0,needs improvement,1,9,32
4,I_kwDOShte388AAAABGT6ozQ,[M0] Cyan — The AI Divide in Computer Studies ...,https://github.com/dataengineeringpilipinas/de...,penOnFire,2026-06-23 00:17:50+08:00,2026-07-22 00:39:13+08:00,OPEN,webzero13,2026-08-01 00:09:44+08:00,M0,needs improvement,1,29,38
